In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/application_train_clean.csv")
print("Loaded:", df.shape)

Loaded: (307511, 74)


In [2]:
# How much of income goes to repaying this loan?
df["ANNUITY_TO_INCOME"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]

# How large is the loan relative to income?
df["CREDIT_TO_INCOME"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]

# How large is the loan vs the goods price? (overpaying = risk signal)
df["CREDIT_TO_GOODS"] = df["AMT_CREDIT"] / df["AMT_GOODS_PRICE"]

print(df[["ANNUITY_TO_INCOME", "CREDIT_TO_INCOME", "CREDIT_TO_GOODS"]].describe().round(3))

       ANNUITY_TO_INCOME  CREDIT_TO_INCOME  CREDIT_TO_GOODS
count         307511.000        307511.000       307511.000
mean               0.181             3.958            1.123
std                0.095             2.690            0.126
min                0.000             0.005            0.150
25%                0.115             2.019            1.000
50%                0.163             3.265            1.119
75%                0.229             5.160            1.198
max                1.876            84.737            6.000


In [3]:
# Convert age from days to years (DAYS_BIRTH is negative, so we negate it)
df["AGE_YEARS"] = (-df["DAYS_BIRTH"]) / 365

# Employment length in years (already fixed, 0 = unemployed)
df["EMPLOYED_YEARS"] = df["DAYS_EMPLOYED"] / 365

# Employment-to-age ratio — how much of adult life spent employed?
df["EMPLOYMENT_RATIO"] = df["EMPLOYED_YEARS"] / (df["AGE_YEARS"] - 18).clip(lower=1)

print(df[["AGE_YEARS", "EMPLOYED_YEARS", "EMPLOYMENT_RATIO"]].describe().round(3))

        AGE_YEARS  EMPLOYED_YEARS  EMPLOYMENT_RATIO
count  307511.000      307511.000        307511.000
mean       43.937          -5.356            -0.253
std        11.956           6.321             0.254
min        20.518         -49.074            -1.019
25%        34.008          -7.562            -0.390
50%        43.151          -3.323            -0.176
75%        53.923          -0.792            -0.042
max        69.121           0.000             0.000


In [4]:
# A ₹1L income means very different things for 1 person vs 5 people
df["INCOME_PER_PERSON"] = df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"]

# Children as a financial burden indicator
df["CHILDREN_RATIO"] = df["CNT_CHILDREN"] / df["CNT_FAM_MEMBERS"].clip(lower=1)

print(df[["INCOME_PER_PERSON", "CHILDREN_RATIO"]].describe().round(3))

       INCOME_PER_PERSON  CHILDREN_RATIO
count       3.075110e+05      307511.000
mean        9.310634e+04           0.126
std         1.013733e+05           0.200
min         2.812500e+03           0.000
25%         4.725000e+04           0.000
50%         7.500000e+04           0.000
75%         1.125000e+05           0.333
max         3.900000e+07           0.950


In [5]:
# Do our new features actually separate defaulters from non-defaulters?
new_features = [
    "ANNUITY_TO_INCOME",
    "CREDIT_TO_INCOME", 
    "CREDIT_TO_GOODS",
    "EMPLOYMENT_RATIO",
    "INCOME_PER_PERSON",
    "CHILDREN_RATIO"
]

summary = df.groupby("TARGET")[new_features].mean().round(3)
print(summary)
print("\n↑ Every column should show a clear difference between 0 and 1")

        ANNUITY_TO_INCOME  CREDIT_TO_INCOME  CREDIT_TO_GOODS  \
TARGET                                                         
0                   0.181             3.964            1.120   
1                   0.185             3.887            1.152   

        EMPLOYMENT_RATIO  INCOME_PER_PERSON  CHILDREN_RATIO  
TARGET                                                       
0                 -0.254          93303.784           0.124  
1                 -0.237          90857.951           0.140  

↑ Every column should show a clear difference between 0 and 1


In [6]:
df.to_csv("data/application_train_features.csv", index=False)
print(f"✅ Saved with {df.shape[1]} total columns")

✅ Saved with 82 total columns


In [7]:
# The issue: DAYS_EMPLOYED after our fix has 0 for unemployed
# and positive values for employed — but very large raw numbers
# Let's inspect what's actually in there

print("DAYS_EMPLOYED sample values:")
print(df["DAYS_EMPLOYED"].describe().round(2))

print("\nEMPLOYED_YEARS sample values:")
print(df["EMPLOYED_YEARS"].describe().round(2))

DAYS_EMPLOYED sample values:
count    307511.00
mean      -1954.85
std        2307.07
min      -17912.00
25%       -2760.00
50%       -1213.00
75%        -289.00
max           0.00
Name: DAYS_EMPLOYED, dtype: float64

EMPLOYED_YEARS sample values:
count    307511.00
mean         -5.36
std           6.32
min         -49.07
25%          -7.56
50%          -3.32
75%          -0.79
max           0.00
Name: EMPLOYED_YEARS, dtype: float64


In [8]:
# DAYS_EMPLOYED is stored as negative in raw data (days before application)
# Our earlier fix replaced 365243 with 0, but kept negatives for employed people
# So we need to take absolute value first

df["EMPLOYED_YEARS"] = df["DAYS_EMPLOYED"].abs() / 365
df["AGE_YEARS"] = (-df["DAYS_BIRTH"]) / 365

# Working years = age minus 18 (earliest working age)
df["WORKING_YEARS_POSSIBLE"] = (df["AGE_YEARS"] - 18).clip(lower=1)
df["EMPLOYMENT_RATIO"] = df["EMPLOYED_YEARS"] / df["WORKING_YEARS_POSSIBLE"]

# Cap at 1.0 — can't be employed more than 100% of working life
df["EMPLOYMENT_RATIO"] = df["EMPLOYMENT_RATIO"].clip(upper=1.0)

# Verify the fix
summary = df.groupby("TARGET")["EMPLOYMENT_RATIO"].mean().round(3)
print("EMPLOYMENT_RATIO by default status:")
print(summary)
print("\n↑ Non-defaulters (0) should have HIGHER employment ratio")

EMPLOYMENT_RATIO by default status:
TARGET
0    0.254
1    0.237
Name: EMPLOYMENT_RATIO, dtype: float64

↑ Non-defaulters (0) should have HIGHER employment ratio


In [9]:
df.to_csv("data/application_train_features.csv", index=False)
print(f"✅ Re-saved with fixed EMPLOYMENT_RATIO — {df.shape[1]} columns")

✅ Re-saved with fixed EMPLOYMENT_RATIO — 83 columns


In [12]:
# Load from the ORIGINAL raw file — before our 40% missing drop
df_original = pd.read_csv("data/application_train.csv")

print("EXT_SOURCE_1 missing %:", df_original["EXT_SOURCE_1"].isnull().mean().round(3))

# Add to our feature dataframe using the index to align rows correctly
df["EXT_SOURCE_1"] = df_original["EXT_SOURCE_1"].values
df["EXT_SOURCE_1_MISSING"] = df["EXT_SOURCE_1"].isnull().astype(int)
df["EXT_SOURCE_1"] = df["EXT_SOURCE_1"].fillna(df["EXT_SOURCE_1"].median())

print("\nEXT_SOURCE_1 by default status:")
print(df.groupby("TARGET")["EXT_SOURCE_1"].mean().round(3))
print("↑ Should be clearly different between 0 and 1")

EXT_SOURCE_1 missing %: 0.564

EXT_SOURCE_1 by default status:
TARGET
0    0.508
1    0.458
Name: EXT_SOURCE_1, dtype: float64
↑ Should be clearly different between 0 and 1


In [13]:
from sklearn.preprocessing import LabelEncoder

# These categoricals are predictive — encode them as numbers
cat_cols_to_add = [
    "NAME_CONTRACT_TYPE",    # Cash vs Revolving loans
    "CODE_GENDER",           # Gender
    "NAME_INCOME_TYPE",      # Working / Pensioner / Commercial etc.
    "NAME_EDUCATION_TYPE",   # Education level
    "NAME_FAMILY_STATUS",    # Married / Single etc.
    "OCCUPATION_TYPE",       # Job type
]

le = LabelEncoder()
for col in cat_cols_to_add:
    df[col + "_ENC"] = le.fit_transform(df[col].astype(str))
    print(f"✅ Encoded {col}")

✅ Encoded NAME_CONTRACT_TYPE
✅ Encoded CODE_GENDER
✅ Encoded NAME_INCOME_TYPE
✅ Encoded NAME_EDUCATION_TYPE
✅ Encoded NAME_FAMILY_STATUS
✅ Encoded OCCUPATION_TYPE


In [14]:
# Number of documents provided — more docs = more verifiable = lower risk
doc_cols = [c for c in df.columns if c.startswith("FLAG_DOCUMENT")]
df["DOCUMENT_COUNT"] = df[doc_cols].sum(axis=1)

# Contact reachability — can lender reach this person?
df["CONTACT_REACHABILITY"] = (
    df["FLAG_MOBIL"] +
    df["FLAG_EMP_PHONE"] +
    df["FLAG_WORK_PHONE"] +
    df["FLAG_PHONE"] +
    df["FLAG_EMAIL"]
)

print(df.groupby("TARGET")[["DOCUMENT_COUNT", "CONTACT_REACHABILITY"]].mean().round(3))

        DOCUMENT_COUNT  CONTACT_REACHABILITY
TARGET                                      
0                0.928                 2.352
1                0.950                 2.418


In [15]:
df.to_csv("data/application_train_features.csv", index=False)
print(f"✅ Re-saved — {df.shape[1]} total columns")

✅ Re-saved — 93 total columns
